In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!tar -xf /content/drive/MyDrive/DNN_Dataset/vit_crops_224.tar -C /content

In [3]:
!pip install -q timm scikit-learn

In [4]:
import json
import random
from pathlib import Path
import shutil

import numpy as np
import torch
import torch.nn as nn
import timm
from tqdm.auto import tqdm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

In [ ]:

ROOT     = Path("/content/vit_crops_224")
TRAIN_DIR = ROOT / "train"
VAL_DIR   = ROOT / "val"
TEST_DIR  = ROOT / "test"

SAVE_DIR = Path("/content/swin_b_classifier")
SAVE_DIR.mkdir(parents=True, exist_ok=True)


NUM_CLASSES      = 5
IMG_SIZE         = 224
BATCH_SIZE       = 32
GRAD_ACCUM_STEPS = 1   
EPOCHS_HEAD      = 2   
EPOCHS_FINE      = 10  
LR_HEAD          = 1e-3
LR_FINE          = 3e-5
WEIGHT_DECAY     = 1e-4
NUM_WORKERS      = 4
SEED             = 42


DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Device:", DEVICE)


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)

Device: cuda


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=eval_tfms)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=eval_tfms)

print("Classes:", train_ds.classes)
print("Class to idx:", train_ds.class_to_idx)


idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}

with open(SAVE_DIR / "class_to_idx.json", "w") as f:
    json.dump(train_ds.class_to_idx, f, indent=4)

with open(SAVE_DIR / "idx_to_class.json", "w") as f:
    json.dump(idx_to_class, f, indent=4)


targets       = torch.tensor(train_ds.targets)
class_counts  = torch.bincount(targets)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

print("Class counts:",  class_counts.tolist())
print("Class weights:", class_weights.tolist())


_loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

train_loader = DataLoader(train_ds, sampler=sampler, **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False,   **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False,   **_loader_kwargs)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: ['boat', 'buoy', 'jetski', 'life_saving_appliances', 'swimmer']
Class to idx: {'boat': 0, 'buoy': 1, 'jetski': 2, 'life_saving_appliances': 3, 'swimmer': 4}
Class counts: [12871, 4239, 2179, 773, 36937]
Class weights: [7.769404328428209e-05, 0.00023590469209011644, 0.0004589261079672724, 0.0012936610728502274, 2.7073125238530338e-05]
Train: 56999 | Val: 9630 | Test: 750


In [8]:
def create_swin_b(num_classes: int = 5) -> nn.Module:
    """Load ImageNet-22k → 1k fine-tuned Swin-B; fall back to IN-1k weights."""
    primary  = "swin_base_patch4_window7_224.ms_in22k_ft_in1k"
    fallback = "swin_base_patch4_window7_224"
    try:
        model = timm.create_model(primary, pretrained=True, num_classes=num_classes)
        print("Loaded:", primary)
    except Exception as e:
        print(f"Primary model failed ({e}); using fallback.")
        model = timm.create_model(fallback, pretrained=True, num_classes=num_classes)
        print("Loaded:", fallback)
    return model


model     = create_swin_b(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loaded: swin_base_patch4_window7_224.ms_in22k_ft_in1k


In [9]:
@torch.no_grad()
def evaluate(model, loader, criterion, split_name: str = "val") -> dict:
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average=None,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )

    return {
        "split":               split_name,
        "loss":                total_loss / len(loader.dataset),
        "accuracy":            accuracy_score(all_labels, all_preds),
        "macro_f1":            f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1":         f1_score(all_labels, all_preds, average="weighted"),
        "precision_per_class": precision,
        "recall_per_class":    recall,
        "f1_per_class":        f1,
        "support_per_class":   support,
        "preds":               all_preds,
        "labels":              all_labels,
    }


def print_results(results: dict, class_names: list) -> None:
    print(f"\n===== {results['split'].upper()} RESULTS =====")
    print(f"Loss:        {results['loss']:.4f}")
    print(f"Accuracy:    {results['accuracy']:.4f}")
    print(f"Macro F1:    {results['macro_f1']:.4f}")
    print(f"Weighted F1: {results['weighted_f1']:.4f}")
    print("\nPer-class metrics:")
    for i, cls in enumerate(class_names):
        print(
            f"{cls:25s} | "
            f"P: {results['precision_per_class'][i]:.4f} | "
            f"R: {results['recall_per_class'][i]:.4f} | "
            f"F1: {results['f1_per_class'][i]:.4f} | "
            f"Support: {results['support_per_class'][i]}"
        )

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch: int, grad_accum_steps: int = 1) -> tuple:
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels) / grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * grad_accum_steps * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | Macro F1: {macro_f1:.4f}")
    return avg_loss, acc, macro_f1


def save_checkpoint(model, epoch: int, stage: str, macro_f1: float, path) -> None:
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_to_idx":     train_ds.class_to_idx,
        "idx_to_class":     idx_to_class,
        "val_macro_f1":     macro_f1,
        "epoch":            epoch,
        "stage":            stage,
    }, path)
    print(f"\u2705 Saved best model ({stage}): {path} | Macro F1: {macro_f1:.4f}")

In [ ]:
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_macro_f1 = -1.0
best_path     = SAVE_DIR / "best_swin_b.pth"

for epoch in tqdm(range(1, EPOCHS_HEAD + 1), desc="Stage 1 – Head"):
    train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch, GRAD_ACCUM_STEPS)
    val_results = evaluate(model, val_loader, criterion, split_name="val")
    print_results(val_results, train_ds.classes)

    if val_results["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_results["macro_f1"]
        save_checkpoint(model, epoch, "head", best_macro_f1, best_path)

Stage 1 – Head:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.1243 | Acc: 0.9617 | Macro F1: 0.9617

===== VAL RESULTS =====
Loss:        0.1264
Accuracy:    0.9548
Macro F1:    0.9000
Weighted F1: 0.9591

Per-class metrics:
boat                      | P: 0.9835 | R: 0.9697 | F1: 0.9766 | Support: 2214
buoy                      | P: 0.9503 | R: 0.9893 | F1: 0.9694 | Support: 560
jetski                    | P: 0.8071 | R: 0.9938 | F1: 0.8908 | Support: 320
life_saving_appliances    | P: 0.5354 | R: 0.9848 | F1: 0.6937 | Support: 330
swimmer                   | P: 0.9980 | R: 0.9428 | F1: 0.9696 | Support: 6206
✅ Saved best model (head): /content/swin_b_classifier/best_swin_b.pth | Macro F1: 0.9000
Epoch 02 | Loss: 0.0629 | Acc: 0.9800 | Macro F1: 0.9801

===== VAL RESULTS =====
Loss:        0.0691
Accuracy:    0.9747
Macro F1:    0.9347
Weighted F1: 0.9755

Per-class metrics:
boat                      | P: 0.9899 | R: 0.9756 | F1: 0.9827 | Support: 2214
buoy                      | P: 0.9669 | R: 0.9911 | F1: 0.9788 | Support: 56

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINE)

start_epoch = EPOCHS_HEAD + 1
end_epoch   = EPOCHS_HEAD + EPOCHS_FINE

for epoch in tqdm(range(start_epoch, end_epoch + 1), desc="Stage 2 – Fine-tune"):
    train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch, GRAD_ACCUM_STEPS)
    val_results = evaluate(model, val_loader, criterion, split_name="val")
    print_results(val_results, train_ds.classes)
    scheduler.step()

    if val_results["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_results["macro_f1"]
        save_checkpoint(model, epoch, "fine_tune", best_macro_f1, best_path)

Stage 2 – Fine-tune:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 03 | Loss: 0.0329 | Acc: 0.9895 | Macro F1: 0.9895

===== VAL RESULTS =====
Loss:        0.0301
Accuracy:    0.9920
Macro F1:    0.9792
Weighted F1: 0.9921

Per-class metrics:
boat                      | P: 0.9982 | R: 0.9955 | F1: 0.9968 | Support: 2214
buoy                      | P: 0.9894 | R: 1.0000 | F1: 0.9947 | Support: 560
jetski                    | P: 0.9786 | R: 1.0000 | F1: 0.9892 | Support: 320
life_saving_appliances    | P: 0.8876 | R: 0.9576 | F1: 0.9213 | Support: 330
swimmer                   | P: 0.9968 | R: 0.9915 | F1: 0.9941 | Support: 6206
✅ Saved best model (fine_tune): /content/swin_b_classifier/best_swin_b.pth | Macro F1: 0.9792
Epoch 04 | Loss: 0.0154 | Acc: 0.9955 | Macro F1: 0.9955

===== VAL RESULTS =====
Loss:        0.0306
Accuracy:    0.9928
Macro F1:    0.9814
Weighted F1: 0.9929

Per-class metrics:
boat                      | P: 0.9991 | R: 0.9932 | F1: 0.9961 | Support: 2214
buoy                      | P: 0.9947 | R: 0.9982 | F1: 0.9964 | Suppor

In [13]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
print("Loaded best model from:", best_path)
print("Best val macro F1:", checkpoint["val_macro_f1"])

test_results = evaluate(model, test_loader, criterion, split_name="test")
print_results(test_results, train_ds.classes)

print("\nClassification report:")
print(classification_report(
    test_results["labels"], test_results["preds"],
    target_names=train_ds.classes,
    zero_division=0,
))

cm = confusion_matrix(test_results["labels"], test_results["preds"])
print("\nConfusion matrix:")
print(cm)

results_to_save = {
    "test_loss":       float(test_results["loss"]),
    "test_accuracy":   float(test_results["accuracy"]),
    "test_macro_f1":   float(test_results["macro_f1"]),
    "test_weighted_f1": float(test_results["weighted_f1"]),
    "classes":         train_ds.classes,
    "confusion_matrix": cm.tolist(),
}

with open(SAVE_DIR / "test_results.json", "w") as f:
    json.dump(results_to_save, f, indent=4)

print(f"\u2705 Test results saved to: {SAVE_DIR / 'test_results.json'}")

Loaded best model from: /content/swin_b_classifier/best_swin_b.pth
Best val macro F1: 0.9895062887225711

===== TEST RESULTS =====
Loss:        0.0564
Accuracy:    0.9920
Macro F1:    0.9920
Weighted F1: 0.9920

Per-class metrics:
boat                      | P: 1.0000 | R: 0.9867 | F1: 0.9933 | Support: 150
buoy                      | P: 0.9934 | R: 1.0000 | F1: 0.9967 | Support: 150
jetski                    | P: 0.9934 | R: 1.0000 | F1: 0.9967 | Support: 150
life_saving_appliances    | P: 1.0000 | R: 0.9733 | F1: 0.9865 | Support: 150
swimmer                   | P: 0.9740 | R: 1.0000 | F1: 0.9868 | Support: 150

Classification report:
                        precision    recall  f1-score   support

                  boat       1.00      0.99      0.99       150
                  buoy       0.99      1.00      1.00       150
                jetski       0.99      1.00      1.00       150
life_saving_appliances       1.00      0.97      0.99       150
               swimmer       0.97 

In [ ]:
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/DNN_Dataset/SeaDronesSee_SwinB")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

drive_best_path = DRIVE_SAVE_DIR / "best_swin_b.pth"

if not best_path.exists():
    raise FileNotFoundError(f"Best model not found: {best_path}")

shutil.copy2(best_path, drive_best_path)

print("✅ Best model saved to Google Drive:")
print(drive_best_path)

✅ Best model saved to Google Drive:
/content/drive/MyDrive/DNN_Dataset/SeaDronesSee_SwinB/best_swin_b.pth
